# 03 · Benchmarking the pipeline on real catalog stars

Two known planets prove the pipeline *can* work. This notebook measures **how well it works** on stars it has never seen, using the **TESS Objects of Interest (TOI) catalog**, which lists thousands of TESS signals and what astronomers concluded about each.

The story in four parts:

1. **A 100-star benchmark** with one sector per star: how often does the pipeline recover the catalog's period?
2. **Diagnosing the misses**, which fall into a few clear groups
3. **The fix:** more data (up to three sectors) raises recovery from **76% to 90%**
4. **Building the training set**: the same pipeline on 1,066 stars, for the classifier in notebook 05

### Re-running vs loading

The batch runs here take from minutes to many hours. With `RERUN = False` (the default), the notebook **loads the saved results** from `data/` instead of reprocessing, so "Restart and Run All" finishes in seconds. Set it to `True` to reprocess from scratch.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings("ignore", message=".*oktopus.*")

import sys
sys.path.append("..")
import pandas as pd
from src.batch import DATA_DIR, load_toi_catalog, select_sample, run_batch, evaluate

RERUN = False   # True = reprocess every batch from scratch (hours); False = load saved results

def run_or_load(targets, name, max_sectors):
    """Run a batch, or load its saved results if they exist and RERUN is off."""
    saved = DATA_DIR / name / "results.csv"
    if RERUN or not saved.exists():
        return run_batch(targets, name=name, max_sectors=max_sectors)
    print(f"Loaded saved results: {saved.relative_to(DATA_DIR.parent)}")
    return pd.read_csv(saved)

---
# Part 1: The 100-star benchmark

## The catalog

Each TOI has a **disposition**: what astronomers concluded after follow-up.

| Code | Meaning |
|---|---|
| **CP** / **KP** | Confirmed planet / known planet |
| **PC** | Planet candidate, not yet resolved |
| **FP** | False positive: a real signal, but not a planet (often an eclipsing binary) |
| **FA** | False alarm: noise or an instrument artifact |
| **APC** | Ambiguous planet candidate |

In [2]:
toi = load_toi_catalog()
print(len(toi), "TOIs")
toi["tfopwg_disp"].value_counts()

8148 TOIs


tfopwg_disp
PC     4815
FP     1313
CP      813
KP      607
APC     487
FA       99
Name: count, dtype: int64

## The sample

`select_sample` picks **50 confirmed planets and 50 false positives**, keeping only:

- stars with a **single** TOI (multi-planet systems would confuse the comparison, since the pipeline reports one signal per star)
- catalog periods of **0.5 to 15 days**, the pipeline's search range
- stars of **TESS magnitude 11 or brighter**, for cleaner data

A fixed random seed makes the sample identical every time.

In [3]:
sample = select_sample(toi)
print(sample["label"].value_counts())
sample.head()

label
planet            50
false_positive    50
Name: count, dtype: int64


,toi,tid,tfopwg_disp,pl_orbper,pl_trandurh,pl_trandep,st_tmag,st_rad,st_logg,st_teff,label,target
0,209.01,52204645,CP,4.379430,1.663935,969.695762,10.85450,0.742779,4.50955,4892.0,planet,TIC 52204645
1,5177.01,350033870,CP,2.208590,3.992000,151.000000,8.84710,1.680000,4.06000,6183.0,planet,TIC 350033870
2,2211.01,212253390,CP,3.092686,2.214920,272.677833,9.25510,0.834386,4.54958,5253.0,planet,TIC 212253390
3,5821.01,354619337,KP,2.142510,5.163000,7961.000000,8.08821,2.020000,4.10500,7893.0,planet,TIC 354619337
4,626.01,65412605,CP,4.401046,4.757000,6903.000000,9.50712,2.320000,4.03000,8489.0,planet,TIC 65412605


## Run it: one sector per star

`run_batch` processes every star with `process_star`, recording errors instead of crashing, and saving each result as it goes so an interrupted run can resume. (A 5-star test run came first, to check everything worked before committing to 100.)

In [4]:
results = run_or_load(sample, "batch1", max_sectors=1)
results["status"].value_counts()

Loaded saved results: data\batch1\results.csv


status
ok       89
error    11
Name: count, dtype: int64

## Scoring the results

`evaluate` compares each recovered period with the catalog's:

- **match**: within 1% of the catalog period
- **alias**: within 1% of 2×, 3×, ½×, or ⅓× the catalog period
- **miss**: anything else
- **below_snr**: SNR under 7, too weak to count even if the period happens to be right

Without the SNR threshold, 42 of 50 planets "matched" (84%). But some of those had no visible transit at all: a lucky period with no real evidence. **With the threshold, the honest figure is 38 of 50 (76%).**

In [5]:
ok, table = evaluate(results)
table

outcome,alias,below_snr,match,miss,All
label,,,,,
false_positive,5,4,26,4,39
planet,1,6,38,5,50
All,6,10,64,9,89


The errors are all stars with **no 2-minute data**: they were found in TESS's full-frame images and never observed at the faster cadence the pipeline uses. Every one is a false positive.

In [6]:
pd.set_option("display.max_colwidth", 80)
results.loc[results["status"] == "error", ["target", "error"]]

,target,error
51,TIC 455962313,ValueError: No TESS SPOC 120s light curves found for 'TIC 455962313'
53,TIC 316606244,ValueError: No TESS SPOC 120s light curves found for 'TIC 316606244'
56,TIC 377420580,ValueError: No TESS SPOC 120s light curves found for 'TIC 377420580'
58,TIC 354600221,ValueError: No TESS SPOC 120s light curves found for 'TIC 354600221'
59,TIC 339538183,ValueError: No TESS SPOC 120s light curves found for 'TIC 339538183'
61,TIC 409183335,ValueError: No TESS SPOC 120s light curves found for 'TIC 409183335'
71,TIC 169177766,ValueError: No TESS SPOC 120s light curves found for 'TIC 169177766'
74,TIC 78672342,ValueError: No TESS SPOC 120s light curves found for 'TIC 78672342'
85,TIC 326114850,ValueError: No TESS SPOC 120s light curves found for 'TIC 326114850'
87,TIC 269424395,ValueError: No TESS SPOC 120s light curves found for 'TIC 269424395'


---
# Part 2: Diagnosing the misses

Every star that wasn't a confident match, alongside the catalog's values. `period_ratio` shows how far off each answer is: about 0.5 or 2 means an alias; anything else means the pipeline found a different signal.

In [7]:
ok = ok.merge(sample[["target", "pl_trandurh", "st_tmag"]], on="target")
ok["period_ratio"] = ok["period_days"] / ok["catalog_period"]

cols = ["target", "label", "outcome", "catalog_period", "period_days", "period_ratio",
        "catalog_depth_ppm", "depth_ppm", "snr", "pl_trandurh", "st_tmag"]
ok[ok["outcome"] != "match"][cols].sort_values(["label", "outcome"])

,target,label,outcome,catalog_period,period_days,period_ratio,catalog_depth_ppm,depth_ppm,snr,pl_trandurh,st_tmag
50,TIC 142918609,false_positive,alias,14.537800,7.278133,0.500635,4040.000000,6864.900734,91.265566,6.826000,9.79250
61,TIC 262781201,false_positive,alias,1.315426,0.658680,0.500735,285.000000,128.863223,7.070243,2.121000,9.34567
70,TIC 306902029,false_positive,alias,1.075457,0.540821,0.502876,690.000000,218.519576,7.808451,1.832000,10.44060
74,TIC 67309890,false_positive,alias,1.162394,0.581320,0.500106,1099.000000,778.310408,17.029733,1.266000,10.21080
79,TIC 136971594,false_positive,alias,1.255620,0.627275,0.499574,290.000000,172.711196,7.710033,2.304000,9.13840
60,TIC 349786918,false_positive,below_snr,2.966339,0.597272,0.201350,192.369144,205.803413,3.397976,4.348679,9.96760
69,TIC 377290815,false_positive,below_snr,0.798000,0.500670,0.627406,190.000000,55.516207,3.322948,0.679000,9.91892
73,TIC 375225453,false_positive,below_snr,0.920953,0.920887,0.999929,360.880153,245.877210,4.674805,1.860273,10.11000
83,TIC 408159788,false_positive,below_snr,2.300698,0.595006,0.258620,780.000000,112.948825,3.572197,3.451000,10.61520
66,TIC 382331352,false_positive,miss,11.476060,0.748769,0.065246,3050.000000,947.495759,32.841773,6.083000,10.00670


In [8]:
ok.groupby(["label", "outcome"])["snr"].median()

label           outcome  
false_positive  alias          7.808451
                below_snr      3.485086
                match         18.417490
                miss         112.764377
planet          alias         37.329981
                below_snr      5.416540
                match         65.989356
                miss          15.282503
Name: snr, dtype: float64

## What the misses showed

The failures fell into four groups:

1. **Eclipsing binaries at half their period.** Every false-positive alias has a ratio of almost exactly 0.500: two eclipses per orbit, read by BLS as one transit at half the period. Not a pipeline bug: telling binaries apart is the classifier's job.
2. **Stellar variability.** Stars whose own brightness swings faster than the 30-hour detrending window can follow. The leftover variability beats the planet, often with a suspiciously high SNR (the SNR assumes random noise, and variability isn't random).
3. **Too few transits.** Long-period planets with only one or two transits in a single 27-day sector, or a period beyond what one sector can search at all.
4. **High-SNR misses on a different signal.** Stars with a second strong periodic signal that outcompetes the catalogued one.

Group 3 has a direct fix: **more data**. The next step reruns every non-match with up to three sectors.

---
# Part 3: More data

## Rerunning the failures with three sectors

The first attempt failed for 5 stars with a surprising error: lightkurve refused to search grids of "15 to 38 million" periods. The traceback showed the number wasn't our grid at all: lightkurve estimates the size of *its own default grid* before every search, even when given a custom one, and that estimate explodes for data spanning years. Two fixes went into `pipeline.py`: a setting that keeps lightkurve's check from firing, and choosing sectors **close together in time** (same number of transits, much smaller grid).

These are the results after both fixes:

In [9]:
redo_targets = ok.loc[ok["outcome"] != "match", "target"]
print(len(redo_targets), "stars to rerun")

rerun = run_or_load(sample[sample["target"].isin(redo_targets)], "redo_3sectors_v2", max_sectors=3)
ok3, table3 = evaluate(rerun)
table3

25 stars to rerun
Loaded saved results: data\redo_3sectors_v2\results.csv


outcome,alias,below_snr,match,miss,All
label,,,,,
false_positive,5,3,1,4,13
planet,0,1,8,3,12
All,5,4,9,7,25


**8 of the 12 failed planets became confident matches.** The false-positive aliases didn't change at all, which confirms their cause is physical (binaries), not a lack of data.

Side by side, with the new reliability flags:

In [10]:
compare = ok[["target", "label", "outcome", "period_days", "snr"]].merge(
    ok3[["target", "outcome", "period_days", "snr", "n_sectors", "sectors","n_transits", "at_search_edge"]],
    on="target", suffixes=("_1sec", "_3sec"))
compare

,target,label,outcome_1sec,period_days_1sec,snr_1sec,outcome_3sec,period_days_3sec,snr_3sec,n_sectors,sectors,n_transits,at_search_edge
0,TIC 52204645,planet,below_snr,4.376547,6.120785,match,4.379450,10.344670,3,1 2 28,15,False
1,TIC 212253390,planet,below_snr,3.091001,5.912500,match,3.092696,7.982707,3,67 92 95,18,False
2,TIC 153976959,planet,below_snr,0.685774,4.979631,below_snr,0.685834,6.407980,3,41 48 49,87,False
3,TIC 98796344,planet,miss,6.531594,30.353606,miss,1.390420,97.485173,2,4 31,28,False
4,TIC 256783784,planet,below_snr,7.403386,5.231021,match,6.109501,12.436620,3,15 16 56,10,False
5,TIC 464646604,planet,miss,11.658259,48.996950,miss,2.257709,144.605907,3,101 103 104,27,False
6,TIC 224225541,planet,alias,9.784280,37.329981,miss,0.882455,39.992963,3,69 96 106,62,False
7,TIC 366484980,planet,below_snr,4.441043,3.099522,match,13.778697,10.609214,3,44 45 46,6,False
8,TIC 263003176,planet,miss,11.470672,11.480255,match,14.356078,20.079228,3,93 94 95,6,False
9,TIC 362249359,planet,below_snr,1.040494,5.602059,match,1.041955,11.357487,3,9 10 11,54,False


## A fair comparison: every star with three sectors

The rerun only gave extra data to stars that *failed*, which makes a "best of two tries" number. For a clean, defensible figure, run **all 100 stars** with the same setting:

In [11]:
results3 = run_or_load(sample, "batch_3sectors", max_sectors=3)
ok_all3, table_all3 = evaluate(results3)
table_all3

Loaded saved results: data\batch_3sectors\results.csv


outcome,alias,below_snr,match,miss,All
label,,,,,
false_positive,6,3,22,8,39
planet,0,1,45,4,50
All,6,4,67,12,89


**45 of 50 confirmed planets recovered at SNR ≥ 7: 90%**, up from 76% with one sector.

Interestingly, false-positive matches went *down* with more data. Real planets become more consistent with more data; impostors (diluted background binaries, one-off systematics) often become less consistent. That's a hint of a future classifier feature.

## The five remaining planet failures

In [12]:
cols = ["target", "catalog_period", "period_days", "catalog_depth_ppm","snr", "sectors", "n_transits", "at_search_edge"]
ok_all3[(ok_all3["label"] == "planet") & (ok_all3["outcome"] != "match")][cols]

,target,catalog_period,period_days,catalog_depth_ppm,snr,sectors,n_transits,at_search_edge
5,TIC 153976959,0.685818,0.685834,148.227758,6.407980,41 48 49,87.0,False
7,TIC 98796344,5.358776,1.390420,2708.439372,97.485173,4 31,28.0,False
18,TIC 464646604,7.712121,2.257709,728.484911,144.605907,101 103 104,27.0,False
22,TIC 224225541,4.937723,0.882455,844.819909,39.992963,69 96 106,62.0,False
46,TIC 333657795,0.926063,12.450407,221.000000,27.302912,42 43 70,1.0,False


These are the pipeline's **known limitations**:

- **One weak detection** with the right period but SNR just under 7: correctly not claimed.
- **One single-event fluke**: `n_transits = 1`. A minimum of 3 transits would reject it automatically.
- **Three stars dominated by stellar variability**, with high SNR on the wrong period. The SNR is overconfident because variability is correlated noise. This motivated `snr_red`, a correlated-noise SNR, among the classifier's features.

---
# Part 4: Building the training set

With the pipeline validated, the same run on **every eligible confirmed planet and false positive** produces the labelled dataset for the classifier: 1,066 stars.

In [13]:
full = select_sample(toi, n_planets=100000, n_fps=100000) # large numbers = take every eligible star
full.to_csv(DATA_DIR / "targets_train_v1.csv", index=False) # also used for the AWS run (notebook 06)
full["label"].value_counts()

label
false_positive    647
planet            419
Name: count, dtype: int64

In [14]:
train = run_or_load(full, "train_v1", max_sectors=3)
print(train["status"].value_counts(), "\n")
print(train.groupby("label")["status"].value_counts(), "\n")
print(train.loc[train["status"] == "error", "error"].str[:45].value_counts())

Loaded saved results: data\train_v1\results.csv
status
ok       933
error    133
Name: count, dtype: int64 

label           status
false_positive  ok        515
                error     132
planet          ok        418
                error       1
Name: count, dtype: int64 

error
ValueError: No TESS SPOC 120s light curves fo    130
ConnectionError: ('Connection aborted.', Remo      2
TimeoutError: Timeout limit of 600 exceeded.       1
Name: count, dtype: int64


**A lesson from this run.** The first overnight attempt reported **460 errors**, mostly false positives, which looked like a major data problem. Grouping the error messages told a different story: **410 were 'ConnectionErrors'**. The laptop had gone to sleep, dropping the network, and the loop kept recording instant failures. Those rows were removed and retried:

```python
retry = res["error"].str.startswith(("ConnectionError", "TimeoutError"), na=False)
res[~retry].to_csv(res_path, index=False)
```

**Always check error types before interpreting error counts.**

The final dataset: **933 stars processed** (418 planets, 515 false positives). The remaining errors are genuine "no 2-minute data" cases, about 20% of false positives, plus a few network failures.

---
# Summary

| Run | Stars | Planets recovered (SNR ≥ 7) |
|---|---|---|
| 1 sector | 100 | 38 / 50 (76%) |
| Up to 3 sectors | 100 | **45 / 50 (90%)** |
| Training set, up to 3 sectors | 1,066 | 418 planets processed; see notebook 06 for the full-sample recovery rate |

**What changed in the pipeline because of this notebook:** sector selection by proximity in time, the lightkurve grid-check fix, the reliability flags (`n_transits`, `at_search_edge`), and the correlated-noise SNR.